# Detecting Metastatic Cancer in Histopathology Patches — Prototype (50k subset)
**Deep Learning PBL · Saif Khattab (202312556) · Al-Zaytoonah University of Jordan**

End-to-end prototype on a **stratified 50,000-image subset** to validate the full pipeline before scaling to all 220,025 images:

1. Setup & data loading
2. Exploratory Data Analysis (EDA)
3. Stratified subset + 70/15/15 split
4. Image loading & quality check (blank / background patches)
5. `tf.data` pipeline with augmentation (no cropping — the label depends on the central 32×32 region)
6. Model 1 — Baseline CNN (from scratch)
7. Model 2 — Transfer learning (EfficientNetB0: frozen head → fine-tuning)
8. Evaluation (ROC-AUC, Sensitivity, Specificity, Precision, F1, Confusion Matrix) + clinical threshold selection
9. Grad-CAM explainability & error analysis

> **Before running:** Settings → Accelerator → **GPU T4 x2**, and Settings → **Turn on internet** (needed to download the ImageNet weights for EfficientNetB0).

> To scale to the full dataset later, set `SUBSET_SIZE = None` (the in-memory loading step will then need ~6 GB RAM for images).

## 1. Setup

In [ ]:
import os, glob, time, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Configuration ----------------
SEED          = 42
SUBSET_SIZE   = 50_000      # None -> use all 220,025 labelled images
IMG_SIZE      = 96
BATCH_SIZE    = 128
EPOCHS_BASE   = 20          # upper bound; EarlyStopping decides
EPOCHS_HEAD   = 5           # transfer learning, stage 1 (frozen base)
EPOCHS_FT     = 15          # transfer learning, stage 2 (fine-tuning)
HEAD_LR       = 1e-3
FT_LR         = 1e-5
N_UNFREEZE    = 60          # number of top EfficientNet layers to unfreeze in stage 2
TARGET_RECALL = 0.95        # clinical operating point: sensitivity we want to guarantee
OUT_DIR       = '/kaggle/working'

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('GPUs      :', tf.config.list_physical_devices('GPU'))

## 2. Locate the data & EDA on the full label file

In [ ]:
candidates = ['/kaggle/input/histopathologic-cancer-detection',
              '/kaggle/input/competitions/histopathologic-cancer-detection']
DATA_DIR = next((d for d in candidates if os.path.exists(os.path.join(d, 'train_labels.csv'))), None)
if DATA_DIR is None:   # fallback: search two levels deep
    found = glob.glob('/kaggle/input/*/train_labels.csv') + glob.glob('/kaggle/input/*/*/train_labels.csv')
    assert found, 'train_labels.csv not found - check that the competition data is attached (Input panel).'
    DATA_DIR = os.path.dirname(found[0])
TRAIN_DIR = os.path.join(DATA_DIR, 'train')

df = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv'))
print('Data dir      :', DATA_DIR)
print('Labelled rows :', len(df))
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate ids :', int(df['id'].duplicated().sum()))
df.head()

In [ ]:
counts = df['label'].value_counts().sort_index()
print(pd.DataFrame({'count': counts, 'percent': (counts / len(df) * 100).round(2)}))

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(['0 - No tumour', '1 - Tumour'], counts.values, color=['#4C72B0', '#DD8452'])
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,}\n({v/len(df):.1%})', ha='center', va='bottom')
ax.set_title('Class distribution (all labelled images)')
ax.set_ylabel('Number of patches'); ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout(); plt.show()

## 3. Stratified 50k subset and 70 / 15 / 15 split

In [ ]:
if SUBSET_SIZE:
    df_sub, _ = train_test_split(df, train_size=SUBSET_SIZE, stratify=df['label'], random_state=SEED)
else:
    df_sub = df.copy()
df_sub = df_sub.reset_index(drop=True)

train_df, temp_df = train_test_split(df_sub, test_size=0.30, stratify=df_sub['label'], random_state=SEED)
val_df,  test_df  = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)

split_summary = pd.DataFrame({
    name: {'images': len(d), 'positives': int(d['label'].sum()), 'positive %': round(d['label'].mean() * 100, 2)}
    for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]
}).T
print(split_summary)

# Save the split so every experiment uses exactly the same images
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d.to_csv(f'{OUT_DIR}/split_{name}.csv', index=False)

## 4. Load images into memory

In [ ]:
def load_image(img_id):
    with Image.open(os.path.join(TRAIN_DIR, f'{img_id}.tif')) as im:
        return np.asarray(im.convert('RGB'), dtype=np.uint8)

def load_images(ids, workers=16):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        arr = np.stack(list(ex.map(load_image, ids)))
    print(f'  loaded {len(ids):,} images {arr.shape} in {time.time() - t0:.0f}s')
    return arr

print('Train:'); X_train = load_images(train_df['id'].values)
print('Val  :'); X_val   = load_images(val_df['id'].values)
print('Test :'); X_test  = load_images(test_df['id'].values)
y_train = train_df['label'].values.astype('float32')
y_val   = val_df['label'].values.astype('float32')
y_test  = test_df['label'].values.astype('float32')
print(f'RAM used by images: {(X_train.nbytes + X_val.nbytes + X_test.nbytes) / 1e9:.2f} GB')

### 4.1 Sample patches (green box = central 32×32 region that defines the label)

In [ ]:
def draw_center(ax):
    ax.add_patch(Rectangle((31.5, 31.5), 32, 32, fill=False, edgecolor='lime', linewidth=1.5))

def show_samples(X, y, n=8, seed=SEED):
    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4.8))
    for row, label in enumerate([0, 1]):
        idx = rng.choice(np.where(y == label)[0], n, replace=False)
        for ax, i in zip(axes[row], idx):
            ax.imshow(X[i]); draw_center(ax); ax.axis('off')
        axes[row, 0].set_title('Label 0 - no tumour' if label == 0 else 'Label 1 - tumour', loc='left')
    plt.tight_layout(); plt.show()

show_samples(X_train, y_train)

### 4.2 Quality check: blank / background-only / dark patches

In [ ]:
def image_stats(X, chunk=2000):
    means, stds = [], []
    for i in range(0, len(X), chunk):
        c = X[i:i + chunk].reshape(len(X[i:i + chunk]), -1).astype(np.float32)
        means.append(c.mean(1)); stds.append(c.std(1))
    return np.concatenate(means), np.concatenate(stds)

tr_mean, tr_std = image_stats(X_train)
flag_bright = tr_mean > 230      # mostly white background
flag_dark   = tr_mean < 25       # (almost) black / corrupted
flag_flat   = tr_std  < 5        # nearly uniform image
flagged = flag_bright | flag_dark | flag_flat

print(f'Very bright (background): {flag_bright.sum()}')
print(f'Very dark               : {flag_dark.sum()}')
print(f'Nearly uniform          : {flag_flat.sum()}')
print(f'Total flagged           : {flagged.sum()} ({flagged.mean():.2%} of train)')
print('Labels of flagged images:', pd.Series(y_train[flagged]).value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for label, color in [(0, '#4C72B0'), (1, '#DD8452')]:
    axes[0].hist(tr_mean[y_train == label], bins=60, alpha=0.6, label=f'label {label}', color=color)
    axes[1].hist(tr_std[y_train == label],  bins=60, alpha=0.6, label=f'label {label}', color=color)
axes[0].set_title('Mean pixel intensity per patch'); axes[1].set_title('Pixel std per patch')
for a in axes: a.legend()
plt.tight_layout(); plt.show()

if flagged.sum():
    idx = np.where(flagged)[0][:8]
    fig, axes = plt.subplots(1, len(idx), figsize=(2 * len(idx), 2.4))
    for ax, i in zip(np.atleast_1d(axes), idx):
        ax.imshow(X_train[i]); ax.set_title(f'y={int(y_train[i])}, m={tr_mean[i]:.0f}'); ax.axis('off')
    plt.suptitle('Examples of flagged patches'); plt.tight_layout(); plt.show()

# Decision for the prototype: flagged patches are KEPT (they are a small fraction and also occur at
# inference time). Their effect is examined in the error analysis at the end.

## 5. `tf.data` pipeline + augmentation (training only)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def augment(img, label):
    # img arrives as float32 in [0, 255]
    img = img / 255.0
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))   # keeps the centre in place
    # mild colour jitter to simulate stain / scanner variation
    img = tf.image.random_brightness(img, 0.10)
    img = tf.image.random_contrast(img, 0.90, 1.10)
    img = tf.image.random_saturation(img, 0.90, 1.10)
    img = tf.image.random_hue(img, 0.04)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img * 255.0, label

def to_float(img, label):
    return tf.cast(img, tf.float32), label

def make_ds(X, y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(10_000, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(to_float, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(X_train, y_train, training=True)
val_ds   = make_ds(X_val,   y_val,   training=False)
test_ds  = make_ds(X_test,  y_test,  training=False)

# Note: both models receive pixels in [0, 255]. The baseline rescales internally;
# Keras EfficientNet includes its own preprocessing and expects [0, 255] inputs.

# Visual check of augmentation on one image
img0 = tf.cast(X_train[0], tf.float32)
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for ax in axes:
    aug, _ = augment(img0, 0)
    ax.imshow(aug.numpy().astype('uint8')); draw_center(ax); ax.axis('off')
plt.suptitle('Augmentations of one training patch'); plt.tight_layout(); plt.show()

In [ ]:
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train.astype(int))
CLASS_WEIGHT = {0: float(cw[0]), 1: float(cw[1])}
print('Class weights:', CLASS_WEIGHT)

def get_metrics():
    return [keras.metrics.AUC(name='auc'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.BinaryAccuracy(name='accuracy')]

def get_callbacks(weights_path, patience=4):
    return [
        keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
        keras.callbacks.ModelCheckpoint(weights_path, monitor='val_auc', mode='max',
                                        save_best_only=True, save_weights_only=True),
    ]

def plot_history(histories, title):
    hist = {}
    for h in histories:
        for k, v in h.history.items():
            hist.setdefault(k, []).extend(v)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for ax, m in zip(axes, ['loss', 'auc']):
        ax.plot(hist[m], label='train'); ax.plot(hist[f'val_{m}'], label='validation')
        ax.set_title(f'{title} - {m}'); ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)
    if len(histories) > 1:
        for a in axes: a.axvline(len(histories[0].history['loss']) - 0.5, color='gray', ls='--', lw=1)
    plt.tight_layout(); plt.show()

## 6. Model 1 — Baseline CNN (from scratch)

In [ ]:
def conv_block(x, filters, dropout, pool=True, last_name=None):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu', name=last_name)(x)
    x = layers.BatchNormalization()(x)
    if pool:
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(dropout)(x)
    return x

def build_baseline():
    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inp)
    x = conv_block(x, 32,  0.10)                                  # 48x48x32
    x = conv_block(x, 64,  0.20)                                  # 24x24x64
    x = conv_block(x, 128, 0.30)                                  # 12x12x128
    x = conv_block(x, 256, 0.0, pool=False, last_name='last_conv')  # 12x12x256
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inp, out, name='baseline_cnn')

baseline = build_baseline()
baseline.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=get_metrics())
baseline.summary()

In [ ]:
BASE_W = f'{OUT_DIR}/baseline.weights.h5'
t0 = time.time()
h_base = baseline.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_BASE,
                      class_weight=CLASS_WEIGHT, callbacks=get_callbacks(BASE_W), verbose=2)
baseline.load_weights(BASE_W)       # make sure we keep the best epoch
print(f'Baseline training time: {(time.time() - t0) / 60:.1f} min')
plot_history([h_base], 'Baseline CNN')

## 7. Model 2 — Transfer learning with EfficientNetB0

In [ ]:
def build_transfer():
    try:
        base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet',
                                                 input_shape=(IMG_SIZE, IMG_SIZE, 3))
    except Exception as e:
        raise RuntimeError('Could not download ImageNet weights. In the Kaggle editor open '
                           'Settings -> Turn on internet, then re-run this cell.') from e
    x = layers.GlobalAveragePooling2D(name='gap')(base.output)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)
    return keras.Model(base.input, out, name='efficientnetb0_tl'), base

tl_model, tl_base = build_transfer()

# ---- Stage 1: train only the new head (base frozen) ----
for layer in tl_base.layers:
    layer.trainable = False
tl_model.compile(optimizer=keras.optimizers.Adam(HEAD_LR), loss='binary_crossentropy', metrics=get_metrics())
print('Trainable params (stage 1):', int(sum(np.prod(w.shape) for w in tl_model.trainable_weights)))

HEAD_W = f'{OUT_DIR}/tl_head.weights.h5'
t0 = time.time()
h_head = tl_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                      class_weight=CLASS_WEIGHT, callbacks=get_callbacks(HEAD_W, patience=3), verbose=2)
tl_model.load_weights(HEAD_W)
val_auc_head = roc_auc_score(y_val, tl_model.predict(val_ds, verbose=0).ravel())
print(f'Stage 1 done in {(time.time() - t0) / 60:.1f} min - val AUC = {val_auc_head:.4f}')

In [ ]:
# ---- Stage 2: fine-tune the top layers (BatchNorm layers stay frozen) ----
for layer in tl_base.layers[-N_UNFREEZE:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True
tl_model.compile(optimizer=keras.optimizers.Adam(FT_LR), loss='binary_crossentropy', metrics=get_metrics())
print('Trainable params (stage 2):', int(sum(np.prod(w.shape) for w in tl_model.trainable_weights)))

FT_W = f'{OUT_DIR}/tl_finetune.weights.h5'
t0 = time.time()
h_ft = tl_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
                    class_weight=CLASS_WEIGHT, callbacks=get_callbacks(FT_W), verbose=2)
tl_model.load_weights(FT_W)
val_auc_ft = roc_auc_score(y_val, tl_model.predict(val_ds, verbose=0).ravel())
print(f'Stage 2 done in {(time.time() - t0) / 60:.1f} min - val AUC = {val_auc_ft:.4f}')

if val_auc_ft < val_auc_head:      # fine-tuning did not help -> keep stage-1 weights
    print('Fine-tuning did not improve validation AUC; reverting to stage-1 weights.')
    tl_model.load_weights(HEAD_W)

plot_history([h_head, h_ft], 'EfficientNetB0 (dashed line = start of fine-tuning)')

## 8. Evaluation

Model selection uses **validation AUC only**; the test set is used **once**, for the selected model.

Clinical note: in screening, a **False Negative** (missed tumour) is more dangerous than a False Positive.
Besides the default threshold 0.5, we therefore pick the threshold on the **validation set** that reaches
`TARGET_RECALL` sensitivity, and report the specificity cost of that choice on the test set.

In [ ]:
def evaluate(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    prec = tp / (tp + fp) if tp + fp else 0.0
    f1   = 2 * prec * sens / (prec + sens) if prec + sens else 0.0
    return {'threshold': round(float(thr), 4), 'AUC': roc_auc_score(y, p), 'Accuracy': (tp + tn) / len(y),
            'Sensitivity (Recall)': sens, 'Specificity': spec, 'Precision': prec, 'F1': f1,
            'FN (missed tumours)': int(fn), 'FP (false alarms)': int(fp)}

def threshold_for_recall(y, p, target):
    fpr, tpr, thr = roc_curve(y, p)
    idx = np.argmax(tpr >= target)          # first (= highest) threshold reaching the target
    return float(thr[idx])

p_val = {'Baseline CNN': baseline.predict(val_ds, verbose=0).ravel(),
         'EfficientNetB0': tl_model.predict(val_ds, verbose=0).ravel()}
val_table = pd.DataFrame({m: evaluate(y_val, p, 0.5) for m, p in p_val.items()}).T
print('Validation results @ threshold 0.5'); display(val_table.round(4))

plt.figure(figsize=(5, 4.5))
for m, p in p_val.items():
    fpr, tpr, _ = roc_curve(y_val, p)
    plt.plot(fpr, tpr, label=f'{m} (AUC={roc_auc_score(y_val, p):.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate (1 - Specificity)'); plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC curves - validation set'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

BEST_NAME  = max(p_val, key=lambda m: roc_auc_score(y_val, p_val[m]))
best_model = tl_model if BEST_NAME == 'EfficientNetB0' else baseline
GRADCAM_LAYER = 'top_activation' if BEST_NAME == 'EfficientNetB0' else 'last_conv'
CLIN_THR = threshold_for_recall(y_val, p_val[BEST_NAME], TARGET_RECALL)
print(f'Selected model: {BEST_NAME}')
print(f'Threshold for >= {TARGET_RECALL:.0%} sensitivity on validation: {CLIN_THR:.4f}')

In [ ]:
# ---- Final, one-time evaluation on the held-out test set ----
p_test = best_model.predict(test_ds, verbose=0).ravel()
test_table = pd.DataFrame({'default (0.5)': evaluate(y_test, p_test, 0.5),
                           f'clinical (recall>={TARGET_RECALL:.0%} on val)': evaluate(y_test, p_test, CLIN_THR)}).T
print(f'TEST results - {BEST_NAME}'); display(test_table.round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, thr) in zip(axes, [('threshold 0.5', 0.5), (f'clinical threshold {CLIN_THR:.3f}', CLIN_THR)]):
    cm = confusion_matrix(y_test, (p_test >= thr).astype(int), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    ax.set_title(f'Test confusion matrix\n{name}')
plt.tight_layout(); plt.show()

# Save predictions for later analysis / report
pd.DataFrame({'id': val_df['id'].values, 'label': y_val.astype(int),
              **{f'p_{m}': p for m, p in p_val.items()}}).to_csv(f'{OUT_DIR}/val_predictions.csv', index=False)
pd.DataFrame({'id': test_df['id'].values, 'label': y_test.astype(int),
              f'p_{BEST_NAME}': p_test}).to_csv(f'{OUT_DIR}/test_predictions.csv', index=False)
val_table.to_csv(f'{OUT_DIR}/val_results.csv'); test_table.to_csv(f'{OUT_DIR}/test_results.csv')
best_model.save(f'{OUT_DIR}/best_model.keras')
with open(f'{OUT_DIR}/run_config.json', 'w') as f:
    json.dump({'subset_size': SUBSET_SIZE, 'seed': SEED, 'best_model': BEST_NAME,
               'clinical_threshold': CLIN_THR, 'target_recall': TARGET_RECALL}, f, indent=2)
print('Saved: predictions, results tables, best_model.keras, run_config.json')

## 9. Grad-CAM explainability & error analysis
Heat-maps show which regions drove the prediction. For true positives we expect attention on the
**central 32×32 region** (green box). The *centre ratio* is the share of Grad-CAM mass inside that box
(a uniform map would give 1/9 ≈ 0.11).

In [ ]:
# Gradients are taken w.r.t. the LOGIT (pre-sigmoid score), not the probability:
# for confident predictions the sigmoid saturates (p ~ 1.0) and its gradient vanishes,
# which would give empty heat-maps.
out_layer  = best_model.layers[-1]          # final Dense(1, sigmoid)
grad_model = keras.Model(best_model.inputs, [best_model.get_layer(GRADCAM_LAYER).output, out_layer.input])

def gradcam(img):
    x = tf.convert_to_tensor(img[None].astype('float32'))
    with tf.GradientTape() as tape:
        conv_out, feats = grad_model(x, training=False)
        logit = tf.matmul(feats, out_layer.kernel) + out_layer.bias
        score = logit[:, 0]
    pred = tf.sigmoid(logit)
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv_out[0] * weights, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE), method='bilinear')[..., 0].numpy()
    return cam, float(pred[0, 0])

def centre_ratio(cam):
    return float(cam[32:64, 32:64].sum() / (cam.sum() + 1e-8))

pred_test = (p_test >= CLIN_THR).astype(int)
groups = {
    'True positives (most confident)': np.where((y_test == 1) & (pred_test == 1))[0][np.argsort(-p_test[(y_test == 1) & (pred_test == 1)])][:6],
    'False negatives (missed tumours)': np.where((y_test == 1) & (pred_test == 0))[0][np.argsort(p_test[(y_test == 1) & (pred_test == 0)])][:6],
    'False positives (false alarms)':   np.where((y_test == 0) & (pred_test == 1))[0][np.argsort(-p_test[(y_test == 0) & (pred_test == 1)])][:6],
}
for title, idx in groups.items():
    if len(idx) == 0:
        print(f'{title}: none'); continue
    fig, axes = plt.subplots(2, len(idx), figsize=(2.3 * len(idx), 5), squeeze=False)
    for j, i in enumerate(idx):
        cam, p = gradcam(X_test[i])
        axes[0, j].imshow(X_test[i]); draw_center(axes[0, j]); axes[0, j].set_title(f'p={p:.3f}'); axes[0, j].axis('off')
        axes[1, j].imshow(X_test[i]); axes[1, j].imshow(cam, cmap='jet', alpha=0.45); draw_center(axes[1, j])
        axes[1, j].set_title(f'centre={centre_ratio(cam):.2f}'); axes[1, j].axis('off')
    plt.suptitle(f'{title} - {BEST_NAME} (threshold {CLIN_THR:.3f})'); plt.tight_layout(); plt.show()

tp_idx = np.where((y_test == 1) & (pred_test == 1))[0][:200]
ratios = [centre_ratio(gradcam(X_test[i])[0]) for i in tp_idx]
print(f'Mean Grad-CAM centre ratio on {len(tp_idx)} true positives: {np.mean(ratios):.3f} (uniform = 0.111)')

# Are the flagged (blank / background) patches over-represented among the errors?
te_mean, te_std = image_stats(X_test)
te_flag = (te_mean > 230) | (te_mean < 25) | (te_std < 5)
err = pred_test != y_test
print(f'Error rate on flagged test patches : {err[te_flag].mean():.2%} (n={te_flag.sum()})' if te_flag.sum() else 'No flagged test patches')
print(f'Error rate on other test patches   : {err[~te_flag].mean():.2%}')

## 10. Summary & known limitations (prototype)
- Trained on a **50k stratified subset**; final results will be produced on the full 220k dataset.
- **Possible data leakage:** the Kaggle version has no patient/slide IDs, so patches from the same slide can fall
  in both train and test, making scores optimistic. Mitigation to consider: evaluate on the original PCam test split
  (which is split by slide).
- EfficientNetB0's last feature map at 96×96 input is only **3×3**, so Grad-CAM maps are coarse.
- The clinical threshold is chosen on validation data; its sensitivity on the test set may differ slightly from the target.
- The system is an **assistive tool only** — final diagnosis remains with a qualified pathologist.

In [ ]:
print('Run configuration:', {'subset': SUBSET_SIZE, 'train/val/test': (len(X_train), len(X_val), len(X_test))})
print(f'Best model: {BEST_NAME}')
display(test_table.round(4))